# RF Análisis Virtual — Mini Penal A · UP N°8 Piñero (Sionna RT)

**Proyecto**: RFQ Motorola · CP 01/26 · Santa Fe — Concurso Público N°01/26
**Objetivo**: Ejecutar ray-tracing de alta fidelidad sobre la geometría 3D del
Mini Penal A para caracterizar la cobertura celular existente como base del
diseño del sistema integral de inhibición.

> **VERSIÓN**: v1.0 (24-abr-2026)
>
> **Metodología**: pipeline BIS validado contra base de datos propia de
> mediciones RF en establecimientos penitenciarios argentinos (MAE 2.90 dB NLOS).
> Premisas sistémicas 3GPP/ENACOM/ITU-R aplicadas al contexto Piñero
> (ver `docs/CALIBRACION_BASE_DATOS_BIS.md`).

## Input
Repo GitHub: https://github.com/MollytheCatLoca/RFQ_MOTOROLA_IMSI
Carpeta in-runtime: `/content/repo/pipeline_rf/inputs/`
  - `antenas_pinero_enriched.csv` — 156 antenas OpenCellID (radio 5 km)
  - `UP9_RF_manifest.json` — manifest 2724 objetos con bbox + material RF
  - `parametros_calibrados.py` — constantes transferidas de la DB propia BIS
  - `UP9_unidad_alto_perfil.obj` — geometría Blender v18 (referencia)

## Output (storage efímero Colab · bajar con files.download)
`/content/outputs/`:
  - `heatmaps_sionna_pinero.png` — 6 mapas (3 bandas × 2 alturas)
  - `mediciones_sionna_coverage_pinero.csv` — raster espacial completo
  - `sionna_stats_pinero.json` — agregados por banda/altura


## 1 · Setup: Install Sionna + dependencies

In [ ]:
# ============================================================================
# NOTEBOOK VERSION: v1.0 (24-abr-2026) · Mini Penal A · UP8 Piñero
# API Sionna RT 1.x probada · install compatible con Colab Python 3.12
# ============================================================================
print('=' * 70)
print('  RF Análisis Piñero v1.0 · pipeline BIS (MAE 2.9 dB NLOS en DB propia)')
print('=' * 70)

# Instalación (~ 2-3 min en Colab)
!pip install -q sionna-rt
!pip install -q pyproj shapely trimesh mapbox-earcut
!pip install -q --upgrade matplotlib pandas

# Check GPU + versions + API introspection
import subprocess, sys, inspect
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip()
print('GPU:', gpu)
print('Python:', sys.version.split()[0])

import sionna.rt as rt
print('sionna-rt version:', getattr(rt, '__version__', '?'))

from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, RadioMapSolver
print('Imports: load_scene, Transmitter, Receiver, PlanarArray, RadioMapSolver OK')

# Diagnóstico de signatures (detecta drift de API antes de celda 7)
print('\n--- Transmitter.__init__ signature ---')
print(inspect.signature(Transmitter.__init__))
print('\n--- PlanarArray.__init__ signature ---')
print(inspect.signature(PlanarArray.__init__))
print('\n--- RadioMapSolver.__call__ signature ---')
try:
    print(inspect.signature(RadioMapSolver.__call__))
except (ValueError, TypeError) as e:
    print(f'(could not introspect: {e})')

import trimesh
try:
    import mapbox_earcut
    print('\nmapbox_earcut OK — trimesh podrá triangular polígonos')
except ImportError:
    print('\n⚠️  mapbox_earcut NO disponible — celda 5 va a skipear edificios!')

## 2 · Clonar repo desde GitHub + paths

In [ ]:
import os, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Clonar repo público
REPO_URL = 'https://github.com/MollytheCatLoca/RFQ_MOTOROLA_IMSI.git'
REPO_DIR = '/content/repo'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# Paths
DRIVE_BASE = Path(f'{REPO_DIR}/pipeline_rf/inputs')
OUTPUT_BASE = Path('/content/outputs')
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print('Repo clonado OK')
print('Input path:', DRIVE_BASE)
print('Output path:', OUTPUT_BASE)
print('Archivos disponibles:')
for f in sorted(DRIVE_BASE.glob('*')):
    print(f'  {f.name}  ({f.stat().st_size // 1024} KB)')

## 3 · Load datasets Piñero

Piñero (sin campaña local previa a Hito 1) trabaja únicamente con:
- manifest RF derivado de geometría Blender (filtrando bbox vol ≥ 50 m³)
- antenas OpenCellID enriquecidas (catálogo 156 celdas radio 3-5 km)
- ray tracing directo sin extrapolación sintética intermedia


In [ ]:
# Manifest RF Piñero (2724 objetos con bbox + material)
with open(DRIVE_BASE / 'UP9_RF_manifest.json') as f:
    MANIFEST = json.load(f)
print(f'Objetos en manifest: {len(MANIFEST["objects"])}')
print(f'Materiales RF definidos: {list(MANIFEST["rf_materials"].keys())}')
print(f'Escena: {MANIFEST["scene"]["description"]}')
print(f'Dimensiones: {MANIFEST["scene"]["dimensions_m"]}')

# Antenas telco Piñero
ANTENAS = pd.read_csv(DRIVE_BASE / 'antenas_pinero_enriched.csv')
print(f'\nAntenas totales: {len(ANTENAS)}')
print('Por operador:')
print(ANTENAS['operator'].value_counts())
print('\nPor tecnología:')
print(ANTENAS['radio'].value_counts())
ANTENAS.head(3)

## 4 · Coordenadas locales (ENU meters) — origen UP8 Piñero

Sionna trabaja en coordenadas cartesianas locales (metros). Convertimos lat/lon al sistema ENU (East-North-Up) con origen en el centro del Mini Penal A Piñero.


In [ ]:
# Centro UP8 Piñero (centroide Complejo Penitenciario)
ORIGIN_LAT = -33.09270383406942
ORIGIN_LON = -60.80047176165265
M_PER_DEG_LAT = 111320.0
M_PER_DEG_LON = 111320.0 * math.cos(math.radians(ORIGIN_LAT))

def to_enu(lat, lon, altitude=0.0):
    '''Convert lat/lon to local ENU (East, North, Up) in meters.'''
    east = (lon - ORIGIN_LON) * M_PER_DEG_LON
    north = (lat - ORIGIN_LAT) * M_PER_DEG_LAT
    return east, north, altitude

def to_latlon(east, north):
    lat = ORIGIN_LAT + north / M_PER_DEG_LAT
    lon = ORIGIN_LON + east / M_PER_DEG_LON
    return lat, lon

# Test con las antenas más cercanas (top 3)
top3 = ANTENAS.nsmallest(3, 'distance_km_to_up8')
print('Top 3 antenas más cercanas al centroide UP8:')
for _, row in top3.iterrows():
    e, n, _ = to_enu(row['lat'], row['lon'])
    print(f'  {row["operator"]:10s} {row["radio"]:4s}: E={e:+7.1f}m  N={n:+7.1f}m  (d={row["distance_km_to_up8"]:.2f} km)')

## 5 · Derivar edificios estructurales del manifest RF

El manifest tiene 2724 objetos incluyendo detalles (ventanas, puertas, decoraciones). Filtramos por **volumen de bbox ≥ 50 m³** para quedarnos con objetos estructurales grandes (pabellones, muros perimetrales, torres, admin).

Los footprints ya están en coordenadas locales (metros) del compound — no hace falta convertir.


In [ ]:
# Filtrar objetos estructurales por volumen de bbox
MIN_VOLUME_M3 = 50.0   # umbral de volumen mínimo

structural_objects = []
for obj in MANIFEST['objects']:
    bmin, bmax = obj['bbox_min'], obj['bbox_max']
    dx, dy, dz = bmax[0]-bmin[0], bmax[1]-bmin[1], bmax[2]-bmin[2]
    vol = dx * dy * dz
    if vol >= MIN_VOLUME_M3:
        structural_objects.append({
            'id': obj['name'].replace('.', '_').replace(' ', '_'),
            'name': obj['name'],
            'material': obj['rf_material'],
            'bbox_min': bmin,
            'bbox_max': bmax,
            'altura_m': dz,
            'centro_x': (bmin[0]+bmax[0])/2,
            'centro_y': (bmin[1]+bmax[1])/2,
            'footprint_m2': dx * dy,
            'vol_m3': vol,
            'muro_db': 25.0 if obj['rf_material'] == 'concrete' else 15.0,
        })

print(f'Objetos estructurales (vol ≥ {MIN_VOLUME_M3} m³): {len(structural_objects)} de {len(MANIFEST["objects"])}')
print(f'  Por material:')
from collections import Counter
for mat, n in Counter(o['material'] for o in structural_objects).most_common():
    print(f'    {mat:20s} {n:4d}')

# Previewl top 10 por volumen
top10 = sorted(structural_objects, key=lambda o: -o['vol_m3'])[:10]
print(f'\nTop 10 por volumen:')
for o in top10:
    print(f'  {o["name"][:40]:40s} vol={o["vol_m3"]:8.0f} m³  h={o["altura_m"]:5.1f} m  ({o["material"]})')

BUILDINGS = structural_objects

## 6 · Build Mitsuba XML scene

**Approach simplificado respecto a sitio de referencia BIS**: en vez de extruir footprints a PLYs, usamos el OBJ Blender v18 completo como una sola shape. Sionna RT 1.x lo acepta directamente.

Los materiales ITU se sustituyen en Cell 7 después de cargar la escena (patrón BIS validado).

In [ ]:
SCENE_DIR = Path('/content/scene_pinero')
SCENE_DIR.mkdir(exist_ok=True)

# Copiar OBJ + MTL al working dir para que Mitsuba los encuentre
import shutil
shutil.copy(DRIVE_BASE / 'UP9_unidad_alto_perfil.obj', SCENE_DIR / 'geometry.obj')
shutil.copy(DRIVE_BASE / 'UP9_unidad_alto_perfil.mtl', SCENE_DIR / 'geometry.mtl')

# Actualizar la referencia al MTL dentro del OBJ (si el nombre cambió)
obj_text = (SCENE_DIR / 'geometry.obj').read_text()
obj_text = obj_text.replace('UP9_unidad_alto_perfil.mtl', 'geometry.mtl')
(SCENE_DIR / 'geometry.obj').write_text(obj_text)

# XML Mitsuba: BSDF placeholders con type='null' (Sionna RT los reemplaza)
xml_text = '''<?xml version="1.0" encoding="utf-8"?>
<scene version="2.1.0">
  <integrator type="path"/>

  <bsdf type="null" id="mat-itu_concrete"/>
  <bsdf type="null" id="mat-itu_wet_ground"/>
  <bsdf type="null" id="mat-itu_metal"/>
  <bsdf type="null" id="mat-itu_glass"/>

  <shape type="obj" id="mini_penal_A">
    <string name="filename" value="geometry.obj"/>
    <ref name="bsdf" id="mat-itu_concrete"/>
  </shape>
</scene>
'''

scene_xml_path = SCENE_DIR / 'pinero_scene.xml'
scene_xml_path.write_text(xml_text)
print(f'Scene XML: {scene_xml_path}')
print(f'OBJ copiado a: {SCENE_DIR}/geometry.obj')
print(f'Tamaño OBJ: {(SCENE_DIR / "geometry.obj").stat().st_size // 1024} KB')

## 7 · Load scene in Sionna + asignar materiales broadband

In [ ]:
import sionna.rt as rt
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, RadioMapSolver

# Load scene
try:
    scene = load_scene(str(scene_xml_path))
    print(f'Scene loaded OK. Objects: {len(scene.objects)}')
except Exception as e:
    print(f'Error loading scene: {e}')
    raise

# Array configuration — isotropic V-pol (simplified — two-slope calibration corrects off-axis)
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
print('✅ TX/RX arrays configurados (iso V-pol)')

In [ ]:
# Materiales broadband (ITU-R P.2040-3 @ 1 GHz) + scattering_coefficient=0.05
# Idéntico a pipeline BIS validado — calibrados contra IDR con MAE 2.9 dB NLOS
from sionna.rt import RadioMaterial

def get_or_create_material(scene, name, **kwargs):
    '''Idempotente: reusa si ya existe, sino crea.'''
    if name in scene.radio_materials:
        mat = scene.radio_materials[name]
        if 'scattering_coefficient' in kwargs:
            mat.scattering_coefficient = kwargs['scattering_coefficient']
        return mat
    mat = RadioMaterial(name=name, **kwargs)
    scene.add(mat)
    return mat

concrete_bb = get_or_create_material(
    scene, 'concrete_bb',
    relative_permittivity=5.24,
    conductivity=0.0462,
    scattering_coefficient=0.05,
)

wet_ground_bb = get_or_create_material(
    scene, 'wet_ground_bb',
    relative_permittivity=30.0,
    conductivity=0.15,
    scattering_coefficient=0.05,
)

# Reasignar todos los objetos (todo concrete_bb; wet_ground_bb si nombre lo indica)
reassigned = 0
for obj_id, obj in scene.objects.items():
    name_lower = obj_id.lower()
    if 'ground' in name_lower or 'floor' in name_lower or 'suelo' in name_lower:
        obj.radio_material = wet_ground_bb
    else:
        obj.radio_material = concrete_bb
    reassigned += 1

# Remover ITU materials del registro (como BIS validado)
itu_names = [n for n in list(scene.radio_materials.keys()) if n.startswith('itu_')]
removed = []
for name in itu_names:
    try:
        scene.remove(name); removed.append(name)
    except Exception:
        pass

print(f'✅ {reassigned} objetos reasignados (concrete_bb scat=0.05)')
print(f'✅ {len(removed)} ITU removidos')
print(f'   Materiales vivos: {list(scene.radio_materials.keys())}')

## 8 · Configure transmitters (antenas telco) + carrier frequency

**Co-ubicación UMTS/LTE aplicada**: el scraping OpenCellID reporta solo 14 antenas LTE en Piñero (ver F2 README). Asumimos que los eNodeB LTE AR se montan sobre la infraestructura UMTS existente — incluimos UMTS + LTE como candidatos TX para las 3 bandas prioritarias.

In [ ]:
TOP_N_ANTENNAS = 15   # por banda

BAND_FREQ_MHZ = {
    'B28': 780.5, 'B26': 876.5, 'B41': 2593.0, 'B66': 2155.0,
    'B4': 2132.5, 'B25': 1962.5, 'B2': 1960.0, 'B7': 2655.0,
}
# Co-ubicación UMTS/LTE: las 3 bandas LTE prioritarias aceptan UMTS también
BAND_TO_TECH = {
    'B28': ['LTE', 'UMTS'],
    'B26': ['LTE', 'UMTS', 'GSM'],
    'B41': ['LTE', 'UMTS'],
    'B66': ['LTE', 'UMTS'],
    'B4': ['LTE', 'UMTS'],
    'B25': ['LTE'],
    'B2': ['LTE'],
    'B7': ['LTE'],
}
EIRP_DBM = 45.0
H_TX = 25.0

def antennas_for_band(banda):
    techs = BAND_TO_TECH[banda]
    df = ANTENAS[ANTENAS['radio'].isin(techs)].copy()
    df = df.sort_values('distance_km_to_up8').head(TOP_N_ANTENNAS)
    return df

def configure_transmitters(banda):
    for name in list(scene.transmitters.keys()):
        scene.remove(name)
    df = antennas_for_band(banda)
    for _, row in df.iterrows():
        e, n, _ = to_enu(row['lat'], row['lon'])
        tx = Transmitter(name=f"tx_{row['id']}",
                         position=[e, n, H_TX],
                         power_dbm=EIRP_DBM)
        scene.add(tx)
    scene.frequency = BAND_FREQ_MHZ[banda] * 1e6
    return len(df)

# Preview
for b in ['B28', 'B26', 'B41']:
    n = configure_transmitters(b)
    print(f'{b}: {n} transmitters @ {BAND_FREQ_MHZ[b]:.0f} MHz')

## 9 · RadioMapSolver setup + coverage bounds

In [ ]:
rm_solver = RadioMapSolver()

import inspect
try:
    sig = inspect.signature(rm_solver.__call__)
    print('RadioMapSolver.__call__ signature:')
    for name, param in sig.parameters.items():
        print(f'  {name}: {param}')
except Exception as e:
    print(f'Could not introspect: {e}')

# Coverage area: cubrir el Mini Penal A con buffer 50 m
# Usamos el bbox global del manifest (extraído del OBJ en F2)
# scene_stats: bbox_min=(-284,-392,-0.4) bbox_max=(476,342,13.3)
e_min, e_max = -180.0, 180.0    # restricted a la zona del mini penal A (no todo el compound)
n_min, n_max = -180.0, 180.0
print(f'Coverage area: E[{e_min:.0f}, {e_max:.0f}]  N[{n_min:.0f}, {n_max:.0f}]')
print(f'Size: {e_max-e_min:.0f}m × {n_max-n_min:.0f}m')

CELL_SIZE_M = 2.0
RESOLUTION = (int((e_max-e_min)/CELL_SIZE_M), int((n_max-n_min)/CELL_SIZE_M))
print(f'Grid resolution: {RESOLUTION[0]} × {RESOLUTION[1]} cells ({RESOLUTION[0]*RESOLUTION[1]} total)')

## 10 · Test run (1 banda × 1 altura)

Antes del full run (60-80 min), corremos una sola combinación con grid reducido para validar API y detectar errores temprano.

In [ ]:
configure_transmitters('B28')
try:
    rm_test = rm_solver(
        scene=scene,
        max_depth=3,
        cell_size=(CELL_SIZE_M*2, CELL_SIZE_M*2),
        center=((e_min+e_max)/2, (n_min+n_max)/2, 1.5),
        orientation=(0, 0, 0),
        size=(e_max-e_min, n_max-n_min),
        samples_per_tx=int(1e5),
        refraction=True,
        specular_reflection=True,
        diffuse_reflection=False,
    )
    print(f'✅ Test run OK. RadioMap type: {type(rm_test).__name__}')
    print(f'   Attributes (first 15): {[a for a in dir(rm_test) if not a.startswith("_")][:15]}')
    for attr in ['path_gain', 'rss', 'sinr']:
        if hasattr(rm_test, attr):
            val = getattr(rm_test, attr)
            shape = val.shape if hasattr(val, 'shape') else '?'
            print(f'   .{attr}: shape={shape}')
except TypeError as e:
    print(f'❌ API mismatch: {e}')
    print('\n--- Signature real ---')
    print(inspect.signature(rm_solver.__call__))
    raise

## 11 · Full run (6 maps = 3 bandas × 2 alturas · ~60-80 min)

In [ ]:
# Configuración idéntica a pipeline BIS validado (max_depth=7, diffuse=True, 2M samples)
BANDS_TO_COMPUTE = ['B28', 'B26', 'B41']
Z_LEVELS = [1.5, 7.5]

coverage_results = {}

for banda in BANDS_TO_COMPUTE:
    configure_transmitters(banda)
    for z in Z_LEVELS:
        print(f'\n▶ Computing: {banda} @ z={z}m (tx={len(scene.transmitters)})')
        rm = rm_solver(
            scene=scene,
            max_depth=7,
            cell_size=(CELL_SIZE_M, CELL_SIZE_M),
            center=((e_min+e_max)/2, (n_min+n_max)/2, z),
            orientation=(0, 0, 0),
            size=(e_max-e_min, n_max-n_min),
            samples_per_tx=int(2e6),
            refraction=True,
            specular_reflection=True,
            diffuse_reflection=True,
            diffraction=True,
            edge_diffraction=True,
        )
        coverage_results[(banda, z)] = rm
        try:
            print(f'  path_gain shape: {rm.path_gain.shape}')
        except AttributeError:
            print(f'  RadioMap attrs: {[a for a in dir(rm) if not a.startswith("_")][:10]}')
        # Checkpoint
        try:
            np.save(OUTPUT_BASE / f'rm_{banda}_z{z}.npy',
                    rm.rss.numpy() if hasattr(rm, 'rss') and rm.rss is not None else rm.path_gain.numpy())
        except Exception as e:
            print(f'  checkpoint save falló: {e}')

print(f'\n✅ {len(coverage_results)} corridas completadas')

## 12 · Export heatmaps + CSV por punto

In [ ]:
# cm_to_dbm idéntico al pipeline BIS — robusto a shapes 2D/3D
import matplotlib.colors as mcolors

def _to_numpy(arr):
    if hasattr(arr, 'numpy'):
        return arr.numpy()
    return np.array(arr)

def cm_to_dbm(rm, eirp_dbm):
    if hasattr(rm, 'rss') and rm.rss is not None:
        rss = _to_numpy(rm.rss)
        if rss.ndim == 3:
            rss_2d = np.max(rss, axis=0)
        else:
            rss_2d = rss
        rss_safe = np.maximum(rss_2d, 1e-30)
        return 10 * np.log10(rss_safe) + 30
    pg = _to_numpy(rm.path_gain)
    if pg.ndim == 3:
        pg = np.max(pg, axis=0)
    pg_safe = np.maximum(pg, 1e-20)
    return eirp_dbm + 10 * np.log10(pg_safe)

fig, axes = plt.subplots(len(BANDS_TO_COMPUTE), len(Z_LEVELS),
                         figsize=(12, 4*len(BANDS_TO_COMPUTE)),
                         squeeze=False)

for i, banda in enumerate(BANDS_TO_COMPUTE):
    for j, z in enumerate(Z_LEVELS):
        cm = coverage_results[(banda, z)]
        rx_dbm = cm_to_dbm(cm, EIRP_DBM)
        print(f'{banda} @ z={z}m: shape={rx_dbm.shape}, '
              f'min={rx_dbm.min():.1f}, max={rx_dbm.max():.1f}, '
              f'median={np.median(rx_dbm):.1f}dBm')
        ax = axes[i, j]
        im = ax.imshow(rx_dbm, extent=[e_min, e_max, n_min, n_max],
                       origin='lower', cmap='viridis',
                       vmin=-120, vmax=-40)
        ax.set_title(f'Piñero · {banda} @ z={z}m (best-server)')
        ax.set_xlabel('East (m)'); ax.set_ylabel('North (m)')
        # Overlay footprints de edificios estructurales
        for b in BUILDINGS[:30]:
            x0, y0 = b['bbox_min'][0], b['bbox_min'][1]
            dx = b['bbox_max'][0] - b['bbox_min'][0]
            dy = b['bbox_max'][1] - b['bbox_min'][1]
            from matplotlib.patches import Rectangle
            ax.add_patch(Rectangle((x0, y0), dx, dy, fill=False, edgecolor='red',
                                   linewidth=0.5, alpha=0.6))
        plt.colorbar(im, ax=ax, label='Rx dBm')

plt.tight_layout()
plt.savefig(OUTPUT_BASE / 'heatmaps_sionna_pinero.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Heatmap guardado: {OUTPUT_BASE}/heatmaps_sionna_pinero.png')

In [ ]:
rows = []
for (banda, z), cm in coverage_results.items():
    rx_dbm = cm_to_dbm(cm, EIRP_DBM)
    ny, nx = rx_dbm.shape
    for iy in range(ny):
        for ix in range(nx):
            e = e_min + (ix + 0.5) * CELL_SIZE_M
            n = n_min + (iy + 0.5) * CELL_SIZE_M
            lat, lon = to_latlon(e, n)
            rows.append({
                'banda': banda, 'z_m': z,
                'lat': lat, 'lon': lon, 'east_m': e, 'north_m': n,
                'rx_dbm': float(rx_dbm[iy, ix]),
            })

df_sionna = pd.DataFrame(rows)
out_csv = OUTPUT_BASE / 'mediciones_sionna_coverage_pinero.csv'
df_sionna.to_csv(out_csv, index=False)
print(f'Exported: {out_csv}  ({len(df_sionna)} rows)')

# Stats resumen
stats = {
    'scene': 'MiniPenalA_UP8_Pinero',
    'origin': {'lat': ORIGIN_LAT, 'lon': ORIGIN_LON},
    'n_tx_per_band': {b: int(antennas_for_band(b).shape[0]) for b in BANDS_TO_COMPUTE},
    'bands': BANDS_TO_COMPUTE, 'z_levels': Z_LEVELS,
    'cell_size_m': CELL_SIZE_M,
    'grid': RESOLUTION,
    'per_map': {},
    'source_notebook': 'v1.0 basado en pipeline BIS validado (MAE 2.9 dB NLOS validado)',
}
for (banda, z), cm in coverage_results.items():
    rx = cm_to_dbm(cm, EIRP_DBM)
    stats['per_map'][f'{banda}_z{z}'] = {
        'median_dbm': float(np.median(rx)),
        'p25': float(np.percentile(rx, 25)),
        'p75': float(np.percentile(rx, 75)),
        'max': float(np.max(rx)),
        'cells_above_m90_pct': float(100 * (rx > -90).sum() / rx.size),
    }
with open(OUTPUT_BASE / 'sionna_stats_pinero.json', 'w') as f:
    json.dump(stats, f, indent=2)
print(f'Stats: {OUTPUT_BASE}/sionna_stats_pinero.json')

## 13 · Download outputs como ZIP

Colab es storage efímero. Bajá el zip antes de cerrar la sesión.

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/PINERO_RF_outputs.zip'
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', str(OUTPUT_BASE))
print(f'Zip creado: {zip_path}')
!ls -la {OUTPUT_BASE}
print('\n⬇ Bajando zip...')
files.download(zip_path)